# 02. Domain Adaptation via Statistical Jittering

For some visual traits, labeled sake-rice grain images are scarce, while a
much larger dataset of visually similar **edible rice** grains is available.
This notebook adapts edible-rice grain crops so their masked-region color
statistics resemble a small reference set of true sake-rice grains,
expanding the effective training set for that trait.

The transform (see `src/sake_rice_inspection/domain_adaptation.py`) is a
per-channel distribution shift:

$$P_{out} = (P_{in} - Mean_{in}) \times \frac{Std_{target}}{Std_{in}} + Mean_{target}$$

Optionally, the target statistics are **jittered** with small Gaussian noise
per source image, so a batch of transformed outputs ends up with slightly
different statistics from one another rather than all collapsing onto one
identical target.

> **Note on data**: this notebook expects grain crops and masks already
> produced by `01_Preprocessing_Cellpose.ipynb` for both the sake-rice
> reference set and the edible-rice source set. The original NDA-protected
> dataset is not included in this repository — point the paths below at your
> own crops.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np

from sake_rice_inspection.domain_adaptation import (
    compute_target_statistics,
    imread_safe,
    transform_directory,
)
import cv2

## 1. Compute target statistics from the sake-rice reference set

`compute_target_statistics` averages the masked mean/std of every reference
grain crop, giving the color distribution the edible-rice crops will be
shifted toward.

In [ ]:
sake_dir = Path("../outputs/crops/target_trait")
sake_mask_dir = Path("../outputs/crops_mask/target_trait")

def load_pairs(image_dir: Path, mask_dir: Path):
    for img_path in sorted(image_dir.glob("*.jpg")):
        mask_path = mask_dir / (img_path.stem + "_mask.png")
        if not mask_path.exists():
            continue
        image = imread_safe(img_path)
        mask = imread_safe(mask_path, cv2.IMREAD_GRAYSCALE)
        if image is not None and mask is not None:
            yield image, mask

target_mean, target_std = compute_target_statistics(load_pairs(sake_dir, sake_mask_dir))
print(f"Target mean (BGR): {target_mean.round(2)}")
print(f"Target std  (BGR): {target_std.round(2)}")

## 2. Transform the edible-rice source set toward the target statistics

`transform_directory` applies the jittered statistical shift to every grain
crop that has a matching mask, writing the results to `output_dir`.

In [ ]:
edible_dir = Path("../outputs/crops/target_trait_edible")
edible_mask_dir = Path("../outputs/crops_mask/target_trait_edible")
output_dir = Path("../outputs/crops/target_trait_edible_adapted")

saved_count, failed_writes = transform_directory(
    edible_dir, edible_mask_dir,
    target_mean=target_mean, target_std=target_std,
    output_dir=output_dir,
    jitter=True,
    rng=np.random.default_rng(42),
)

print(f"Saved: {saved_count}")
if failed_writes:
    print("Failed writes:", failed_writes[:5])